# Example of inference code

In [ ]:
import pandas as pd
import torch
import numpy as np
from data import patches_extraction
import os
from model import PST_PCQAModule
import time
import matplotlib.pyplot as plt
from data import xyz_1_2001
import open3d as o3d
import matplotlib.cm as cm

path_model_to_test = "best_models/WPC/PST_PCQAModule_K_16.ckpt"

info_file = pd.read_excel("data/WPC/mos.xls")
path_ply = "data/WPC/distorted"
configs = {
        "name": "PST_PCQAModule",
        "number_patches": 16,
        "points_per_patch" : 14900,
        "points_texture" : 8192,
        "points_structure" : 1024,
        "batch_size" : 4,
        "lr" : 0.001,
        "dropout": 0.,
        "epochs": 400
    }


trained_model = PST_PCQAModule.load_from_checkpoint(path_model_to_test,  points_texture = configs["points_texture"],
                                                         points_structure = configs["points_structure"], dropout = configs["dropout"], patches=configs['number_patches'], lr = configs["lr"]).eval()
testing_pointcloud = os.path.join("example_pointcloud/house_gsigma_0_tsigma_16.ply")
x_b, x_s = patches_extraction(filename = testing_pointcloud, number_patches = 16, points_per_patch = 14900, small_points_per_patch = 8192)
# Sampling for the structure patch information
index_b = np.arange(x_b.shape[1])
np.random.seed(0)
torch.manual_seed(1)
index_b = np.random.choice(index_b, configs["points_structure"], replace=False)
x_b = x_b[:, index_b]
x_b, x_s = torch.tensor(np.array(x_b), device=trained_model.device).float(), torch.tensor(np.array(x_s), device=trained_model.device).float()
print("Input texture points: {}".format(x_s.shape))
print("Input structure points: {}".format(x_b.shape))

print("##################################### INFERENCE #########################################")
with torch.no_grad():
    for _ in range(10):    # For evaluating inference time
        _ = trained_model(x_b.unsqueeze(0), x_s.unsqueeze(0))
    start_time = time.time()
    mos, mos_per_patch = trained_model(x_b.unsqueeze(0), x_s.unsqueeze(0))
    end_time = time.time()
    inference_time = end_time - start_time
    print(f"Inference time after warm-up: {inference_time:.4f} seconds")
    
print("Final MOS: {}".format(mos))
print("Patch-wise MOS:  {}".format(mos_per_patch))


import numpy as np

point_cloud_patches = x_s.cpu()

# Normalize RGB values to [0, 1] if they are in the range [0, 255]
point_cloud_patches[..., 3:] = point_cloud_patches[..., 3:] / 255.0

# Create a single 3D plot
fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(111, projection='3d')
colormap = cm.coolwarm
score = mos_per_patch.cpu().numpy()[0]

for i in range(point_cloud_patches.shape[0]):
    # Extract x, y, z coordinates and RGB colors
    x = point_cloud_patches[i, :, 0]
    y = point_cloud_patches[i, :, 1]
    z = point_cloud_patches[i, :, 2]
    colors = point_cloud_patches[i, :, 3:]

     # Get the color for the current patch based on its score
    patch_color = colormap(score[i])
    
    # Plot the point cloud with colors
    ax.scatter(x, y, z,  c=[patch_color], marker='o', label=f'Patch {i+1}', zorder=100)
    centroid_x, centroid_y, centroid_z = torch.mean(x), torch.mean(y), torch.mean(z)
     
    ax.text(centroid_x, centroid_y+120, centroid_z, f'{score[i]:.3f}', color='black', fontsize=10, zorder = 100)

pcd = o3d.io.read_point_cloud(testing_pointcloud)
pcd = pcd.farthest_point_down_sample(20000)
points = np.asarray(pcd.points)                         
colors = np.asarray(pcd.colors)

pc = np.concatenate((points, colors), axis=1)
pc[:,0:3] = xyz_1_2001(pc[:,0:3])      
x = pc[:, 0]
y = pc[:, 1]
z = pc[:, 2]
ax.scatter(x, y, z, c=colors, marker='o', label=f'Entire Pointcloud', zorder=1, alpha=0.1)
ax.set_title(f'Point cloud')
ax.view_init(elev=24, azim=-127, roll= 0)

# Adjust layout for better visualization
plt.tight_layout()
sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=np.min(score), vmax=np.max(score)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.1)
cbar.set_label('Patch Score')
plt.show()
